# Tema 5 · Caso práctico · CiudadMX Energía

### Predicción horaria de demanda con Apache Spark (MLlib)

**Uso sugerido:** Para comprender cómo las técnicas de análisis avanzado con Apache Spark permiten predecir demanda energética y apoyar decisiones operativas, revisa este caso y ejecuta el cuaderno paso a paso.

**Contexto (resumen):** La empresa pública *CiudadMX Energía* requiere predecir la demanda eléctrica por hora usando variables de clima (temperatura, humedad), calendario (hora, día, mes) y festivos. Construiremos un **pipeline de MLlib** con **Gradient-Boosted Trees Regressor (GBTR)** y compararemos brevemente con **Regresión Lineal**.


In [1]:
# 1) Instalación de Spark en Colab (Java + PySpark)
!apt-get -y install openjdk-17-jdk > /dev/null
!pip install -q pyspark==3.5.1
print("✅ PySpark 3.5.1 instalado")

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-17/openjdk-17-jre_17.0.16%2b8%7eus1-0ubuntu1%7e22.04.1_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/o/openjdk-17/openjdk-17-jdk_17.0.16%2b8%7eus1-0ubuntu1%7e22.04.1_amd64.deb  404  Not Found [IP: 185.125.190.81 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
✅ PySpark 3.5.1 instalado


In [2]:
# 2) Sesión Spark e imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.regression import GBTRegressor, LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np, pandas as pd

spark = SparkSession.builder.appName("Tema5_DemandaEnergetica").getOrCreate()
spark

## Generación de datos sintéticos (horarios)
Simulamos 12 semanas (~84 días) de datos por hora con señales de clima y calendario. La variable objetivo es `demanda_mwh`.

In [3]:
# 3) Generar datos sintéticos con Pandas
rng = np.random.default_rng(42)
horas = pd.date_range("2025-05-01", periods=24*84, freq="H")  # 84 días, datos horarios

dfp = pd.DataFrame({"ts": horas})
dfp["dia_semana"] = dfp["ts"].dt.dayofweek        # 0=Lunes ... 6=Domingo
dfp["hora"] = dfp["ts"].dt.hour
dfp["mes"] = dfp["ts"].dt.month

# Señales de clima (sazonalidad diaria + ruido)
dfp["temp_c"] = 22 + 8*np.sin(2*np.pi*(dfp["hora"]-6)/24) + rng.normal(0, 2, len(dfp))
dfp["humedad"] = 55 + 15*np.sin(2*np.pi*(dfp["hora"])/24 + 1) + rng.normal(0, 5, len(dfp))

# Festivo simple: marcamos los domingos
dfp["festivo"] = (dfp["dia_semana"] == 6).astype(int)

# Función no lineal para demanda
demanda = (
    300
    + 25*np.where((dfp["hora"]>=18)&(dfp["hora"]<=22), 1, 0)   # pico nocturno
    + 6*(30 - np.clip(dfp["temp_c"], 10, 35))                   # efecto temperatura
    + 0.8*dfp["humedad"]                                        # efecto humedad
    + 15*np.where(dfp["festivo"]==1, 0.85, 1.0)                 # festivos bajan consumo
    + rng.normal(0, 8, len(dfp))
)
dfp["demanda_mwh"] = np.maximum(50, demanda)

dfp.head()

/tmp/ipython-input-2129144775.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  horas = pd.date_range("2025-05-01", periods=24*84, freq="H")  # 84 días, datos horarios


,ts,dia_semana,hora,mes,temp_c,humedad,festivo,demanda_mwh
0,2025-05-01 00:00:00,3,0,5,14.609434,67.481649,0,466.026776
1,2025-05-01 01:00:00,3,1,5,12.192625,65.988086,0,479.394337
2,2025-05-01 02:00:00,3,2,5,16.572699,72.892548,0,448.081253
3,2025-05-01 03:00:00,3,3,5,18.224275,72.038934,0,444.578795
4,2025-05-01 04:00:00,3,4,5,14.097930,68.403166,0,482.725738


In [4]:
# 4) Convertir a Spark DataFrame y hacer split de entrenamiento/prueba
schema = StructType([
    StructField("ts", TimestampType(), True),
    StructField("dia_semana", IntegerType(), True),
    StructField("hora", IntegerType(), True),
    StructField("mes", IntegerType(), True),
    StructField("temp_c", DoubleType(), True),
    StructField("humedad", DoubleType(), True),
    StructField("festivo", IntegerType(), True),
    StructField("demanda_mwh", DoubleType(), True),
])

sdf = spark.createDataFrame(
    dfp.astype({"dia_semana":"int32","hora":"int32","mes":"int32","festivo":"int32"}),
    schema=schema
)

train, test = sdf.randomSplit([0.8, 0.2], seed=42)
train.count(), test.count()

(1653, 363)

## Pipeline de MLlib (OneHot + Escalado + Ensamble + GBTRegressor)
Entrenamos un **Gradient-Boosted Trees Regressor** (GBTR) y generamos predicciones sobre el conjunto de prueba.

In [ ]:
# 5) Definir y entrenar el pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.regression import GBTRegressor, LinearRegression
from pyspark.ml import Pipeline

idx_dia = StringIndexer(inputCol="dia_semana", outputCol="dia_idx", handleInvalid="keep")
idx_mes = StringIndexer(inputCol="mes", outputCol="mes_idx", handleInvalid="keep")

ohe_dia = OneHotEncoder(inputCols=["dia_idx"], outputCols=["dia_ohe"])
ohe_mes = OneHotEncoder(inputCols=["mes_idx"], outputCols=["mes_ohe"])

assembler = VectorAssembler(
    inputCols=["hora","temp_c","humedad","festivo","dia_ohe","mes_ohe"],
    outputCol="feats_raw"
)

scaler = StandardScaler(inputCol="feats_raw", outputCol="features", withStd=True, withMean=False)

gbt = GBTRegressor(featuresCol="features", labelCol="demanda_mwh",
                   maxDepth=6, maxIter=60, stepSize=0.1, seed=42)

pipeline = Pipeline(stages=[idx_dia, idx_mes, ohe_dia, ohe_mes, assembler, scaler, gbt])
model = pipeline.fit(train)

pred = model.transform(test)
pred.select("ts","demanda_mwh","prediction").show(10, truncate=False)

## Métricas de evaluación (RMSE, MAE, R²)

In [ ]:
# 6) Evaluación del modelo
from pyspark.ml.evaluation import RegressionEvaluator

eval_rmse = RegressionEvaluator(labelCol="demanda_mwh", predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol="demanda_mwh", predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol="demanda_mwh", predictionCol="prediction", metricName="r2")

rmse = eval_rmse.evaluate(pred)
mae  = eval_mae.evaluate(pred)
r2   = eval_r2.evaluate(pred)

print(f"RMSE: {rmse:.3f}  |  MAE: {mae:.3f}  |  R2: {r2:.3f}")

## Visualización: Real vs. Predicho

In [ ]:
# 7) Visualización de predicciones vs. valores reales (muestra ordenada por tiempo)
import matplotlib.pyplot as plt

sample_pd = (pred
  .select("ts","demanda_mwh","prediction")
  .orderBy("ts")
  .limit(300)
  .toPandas()
)

plt.figure(figsize=(12,4))
plt.plot(sample_pd["ts"], sample_pd["demanda_mwh"], label="Real", linewidth=2)
plt.plot(sample_pd["ts"], sample_pd["prediction"], label="Predicho", linewidth=2)
plt.title("Demanda horaria: Real vs. Predicho (muestra)")
plt.xlabel("Tiempo")
plt.ylabel("MWh")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Visualización: Residuales (Error = Real - Predicho)

In [ ]:
# 8) Gráfica de residuales
sample_pd["residual"] = sample_pd["demanda_mwh"] - sample_pd["prediction"]

plt.figure(figsize=(12,3.8))
plt.plot(sample_pd["ts"], sample_pd["residual"], color="#cc5500")
plt.axhline(0, color="black", linewidth=1)
plt.title("Residuales a lo largo del tiempo (muestra)")
plt.xlabel("Tiempo")
plt.ylabel("Error")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.hist(sample_pd["residual"], bins=25, color="#3b82f6", edgecolor="white")
plt.title("Distribución de residuales (muestra)")
plt.xlabel("Error")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## Comparación rápida: Regresión Lineal (opcional)

In [ ]:
# 9) Comparación opcional con Regresión Lineal
lr = LinearRegression(featuresCol="features", labelCol="demanda_mwh", regParam=0.01, elasticNetParam=0.0, maxIter=100)
pipe_lr = Pipeline(stages=[idx_dia, idx_mes, ohe_dia, ohe_mes, assembler, scaler, lr])
model_lr = pipe_lr.fit(train)
pred_lr = model_lr.transform(test)

eval_rmse = RegressionEvaluator(labelCol="demanda_mwh", predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol="demanda_mwh", predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol="demanda_mwh", predictionCol="prediction", metricName="r2")

print(
    "LR  =>",
    "RMSE:", round(eval_rmse.evaluate(pred_lr),3),
    "| MAE:", round(eval_mae.evaluate(pred_lr),3),
    "| R2:",  round(eval_r2.evaluate(pred_lr),3),
)

## Próximos pasos
- Añadir nuevas variables (viento, precio spot, eventos).
- Guardar el modelo (`model.write().save(...)`) y crear un endpoint batch/streaming.
- Llevar las predicciones a un dashboard (Power BI, Grafana) para operación diaria.

In [ ]:
# 10) Finalizar sesión
spark.stop()
print("✅ Spark detenido")